# PGx Risk Calculator Dashboard – Full Deployment Workflow

**Purpose:** Deploy the PGx Risk Calculator Dashboard from cohorts with aggregated feature importances through Lambda/Docker.  
**Updated:** January 2026

## Overview

This notebook **consolidates the pipeline from Step 4 onward**. It **runs pipeline Step 4, 5, and 6 in this notebook** (see cells below), then prepares and deploys the dashboard:

- **Pipeline Step 4** — Model data: `4_model_data/create_model_data.py` (model_events.parquet per cohort/age_band)
- **Pipeline Step 5** — PGx analysis: `5_pgx_analysis/run_analysis.py` (PGx features added to model data)
- **Pipeline Step 6** — Final model training: `6_final_model/run_final_model.py` (trained models and feature_schema.json)

**Required inputs (must exist before running pipeline Step 4):**

- **Feature importances** (Step 3/3b) — synced from S3 or present under project/NVMe  
- **SHAP/FFA outputs** (Steps 7 & 8) — optional, for causal tab

## Cohort / model mapping

| Model | PGx cohort | Age bands | Description |
|-------|------------|-----------|-------------|
| **Opioid ED** | `opioid_ed` | 13-24, 25-44, 45-54, 55-64 | Opioid-related ED visit predictive model |
| **Polypharmacy** | `non_opioid_ed` | 65-74, 75-84, 85-94 | Polypharmacy / adverse drug event model |

## Workflow Steps

1. **Sync inputs from S3 to NVMe** (idempotent) – Step 3a/3b feature importance (and optionally Step 6 if already built elsewhere).
2. **Verify inputs** – Feature importance (Step 3/3b) per cohort/age_band; Step 6 outputs if already present.
3. **Pipeline Step 4–6** – Run the **Pipeline Step 4**, **5**, and **6** cells below (model data → PGx analysis → final model training). Skip if Step 6 outputs already exist and are synced.
4. **Generate metadata** (idempotent, checkpoint) – Extract valid codes from feature importance for dashboard dropdowns.
5. **Prepare models** (idempotent, checkpoint) – Package models and feature schemas from Step 6 outputs.
6. **Combine SHAP/FFA** (optional) – For causal analysis tab.
7. **Prepare Lambda directory** – Assemble `lambda_dir` for Docker build.
8. **Verify & deploy** – Verify `lambda_dir`, then build Docker image and deploy (ECR/API Gateway).

## Reference

- PGx data prep: `9_risk_dashboard/data_preparation/`

In [ ]:
# Setup: paths and project root
import sys
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == "9_risk_dashboard":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif (PROJECT_ROOT / "9_risk_dashboard").exists():
    pass
else:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))
from py_helpers.env_utils import get_data_root
from py_helpers.workflow_sync_checkpoint import sync_s3_to_local, check_step_checkpoint_exists, save_step_checkpoint

DASHBOARD_DIR = PROJECT_ROOT / "9_risk_dashboard"
DATA_PREP_DIR = DASHBOARD_DIR / "data_preparation"
DEPLOY_DIR = DASHBOARD_DIR / "deployment"
S3_BUCKET = os.environ.get("PGX_S3_BUCKET", "pgxdatalake")
DATA_ROOT = get_data_root()
AWS_PROFILE = os.environ.get("AWS_PROFILE")

print("PGx Risk Calculator Workflow")
print("=" * 60)
print(f"Project root: {PROJECT_ROOT}")
print(f"Dashboard dir: {DASHBOARD_DIR}")
print(f"Data prep: {DATA_PREP_DIR}")
print(f"Data root (NVMe/local): {DATA_ROOT}")
print("=" * 60)

In [ ]:
# Configuration: PGx cohorts and age bands (aligned with prepare_lambda_dir.py)
# opioid_ed (younger age bands); non_opioid_ed / polypharmacy (older age bands)
REQUIRED_COHORTS = {
    "opioid_ed": ["13-24", "25-44", "45-54", "55-64"],
    "non_opioid_ed": ["65-74", "75-84", "85-94"],
}

# Feature importance: use DATA_ROOT/gold/feature_importance (NVMe or PGX_DATA_ROOT).
# Sync cell writes here; Step 2 artifacts on EC2 can also write here. One tree for 3a + 3b CSVs.
FI_ROOT = DATA_ROOT / "gold" / "feature_importance"
STEP3_OUTPUTS = STEP3B_OUTPUTS = FI_ROOT
# Step 6 outputs: project; DATA_ROOT/6_final_model/outputs; or DATA_ROOT/gold/final_model (S3-synced, same as feature_importance)
FINAL_MODEL_OUTPUTS = PROJECT_ROOT / "6_final_model" / "outputs"
FINAL_MODEL_OUTPUTS_ALT = DATA_ROOT / "6_final_model" / "outputs"
FINAL_MODEL_GOLD = DATA_ROOT / "gold" / "final_model"  # S3 layout: cohort/13-24/*.joblib (hyphen in age_band)

print("Cohorts and age bands:")
for cohort, bands in REQUIRED_COHORTS.items():
    print(f"  {cohort}: {bands}")
print("\nInput dirs:")
print(f"  Feature importance (3/3b): {STEP3_OUTPUTS}")
print(f"  Step 6 (project):  {FINAL_MODEL_OUTPUTS}")
print(f"  Step 6 (NVMe):    {FINAL_MODEL_OUTPUTS_ALT}")
print(f"  Step 6 (gold/NVMe): {FINAL_MODEL_GOLD}")

## Sync required inputs from S3 to NVMe (idempotent)

Sync Step 3a feature importance, Step 3b outputs, and Step 6 final model outputs from S3 so data preparation can read from local/NVMe. **Idempotent:** `aws s3 sync` only updates changed or missing files.

In [ ]:
# Sync Step 3a/3b feature importance and Step 6 final models from S3 to NVMe (DATA_ROOT).
# Feature importance -> DATA_ROOT/gold/feature_importance; Step 6 -> DATA_ROOT/gold/final_model (same bucket layout).
FI_SYNC_TARGET = DATA_ROOT / "gold" / "feature_importance"
FI_SYNC_TARGET.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_GOLD.mkdir(parents=True, exist_ok=True)

sync_s3_to_local(f"s3://{S3_BUCKET}/gold/feature_importance/", FI_SYNC_TARGET, profile=AWS_PROFILE)
sync_s3_to_local(f"s3://{S3_BUCKET}/gold/final_model/", FINAL_MODEL_GOLD, profile=AWS_PROFILE)
print("Sync complete. Run Step 0 verification below.")

## Step 0: Verify inputs (feature importance)

Ensure feature importance (Step 3/3b) exists for each cohort/age_band so pipeline Step 4 can run. This notebook runs pipeline steps 4→5→6; after they complete, Step 6 outputs are used for "Prepare models" and deployment. If Step 6 outputs already exist (e.g. from a previous run or sync from S3), verification passes and you can skip re-running steps 4–6.

In [ ]:
def check_feature_importance(cohort: str, age_band: str) -> bool:
    ab = age_band.replace("-", "_")
    # Step 3b refined: FI_ROOT (NVMe) then project 3b/outputs
    for base in (STEP3B_OUTPUTS, PROJECT_ROOT / "3b_feature_importance_eda" / "outputs"):
        fi_3b = base / cohort / ab / f"{cohort}_{ab}_cohort_feature_importance.csv"
        if fi_3b.exists():
            return True
    # Step 3 aggregated: FI_ROOT then project 3a/outputs
    for base in (STEP3_OUTPUTS, PROJECT_ROOT / "3a_feature_importance" / "outputs"):
        fi_3 = base / cohort / ab / f"{cohort}_{ab}_aggregated_feature_importance.csv"
        if fi_3.exists():
            return True
    return False

def check_final_model(cohort: str, age_band: str) -> bool:
    ab = age_band.replace("-", "_")
    # 1) Project or DATA_ROOT/6_final_model/outputs: cohort/13_24/models/*.joblib
    for base in (FINAL_MODEL_OUTPUTS, FINAL_MODEL_OUTPUTS_ALT):
        model_dir = base / cohort / ab
        if not model_dir.exists():
            continue
        models_sub = model_dir / "models"
        if models_sub.exists() and any(models_sub.glob("*.joblib")):
            return True
        if (model_dir / "feature_schema.json").exists():
            return True
    # 2) DATA_ROOT/gold/final_model (S3-synced): cohort/13-24/*.joblib (hyphen in age_band)
    gold_dir = FINAL_MODEL_GOLD / cohort / age_band
    if gold_dir.exists() and any(gold_dir.glob("*.joblib")):
        return True
    return False

print("Step 0: Verifying inputs per cohort/age_band\n")
all_ok = True
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        fi_ok = check_feature_importance(cohort, age_band)
        model_ok = check_final_model(cohort, age_band)
        status = "OK" if (fi_ok and model_ok) else "MISSING"
        if not (fi_ok and model_ok):
            all_ok = False
        print(f"  {cohort} / {age_band}:  FI={fi_ok}, Model={model_ok}  -> {status}")
if all_ok:
    print("\nAll inputs present.")
else:
    print("\nFix missing inputs before running data preparation.")
    print("  Model=False: Step 6 (final model) outputs missing.")
    print("  Run pipeline steps 4–6 in this notebook to produce Step 6 outputs, or sync from S3 / ensure 6_final_model/outputs (or NVMe) is populated.")
    print("  Paths checked: 1) " + str(FINAL_MODEL_OUTPUTS) + "  2) " + str(FINAL_MODEL_OUTPUTS_ALT) + "  3) " + str(FINAL_MODEL_GOLD) + " (run sync cell to pull from S3)")
    if FINAL_MODEL_GOLD.exists():
        cohorts_in_gold = sorted(d.name for d in FINAL_MODEL_GOLD.iterdir() if d.is_dir())
        print("  Diagnostic: " + str(FINAL_MODEL_GOLD) + " exists. Cohort dirs: " + (str(cohorts_in_gold) if cohorts_in_gold else "[]"))
        if cohorts_in_gold:
            sample = FINAL_MODEL_GOLD / cohorts_in_gold[0]
            bands = sorted(d.name for d in sample.iterdir() if d.is_dir())
            print("  Sample " + cohorts_in_gold[0] + " age_bands: " + str(bands[:5]) + (" ..." if len(bands) > 5 else ""))
            if bands:
                sample_ab = sample / bands[0]
                joblibs = list(sample_ab.glob("*.joblib"))
                print("  Sample " + cohorts_in_gold[0] + "/" + bands[0] + " *.joblib count: " + str(len(joblibs)))
    else:
        print("  Diagnostic: " + str(FINAL_MODEL_GOLD) + " does not exist. Run the Sync cell above first.")
    print("  Run the Sync cell above to copy s3://.../gold/final_model/ to NVMe, or run Step 6 locally.")

## Pipeline Step 4: Model data

Build `model_events.parquet` for each cohort/age_band from Step 2 cohort data and Step 3b feature importance. Outputs go to `4_model_data/cohort_name={cohort}/age_band={age_band}/model_events.parquet`. Run the cell below for all PGx cohorts/age_bands defined in this notebook.

In [ ]:
# Pipeline Step 4: create_model_data.py for each REQUIRED_COHORTS (cohort, age_band)
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"→ Step 4: {cohort} / {age_band}")
        r = subprocess.run(
            [sys.executable, "create_model_data.py", "--cohort", cohort, "--age-band", age_band],
            cwd=PROJECT_ROOT / "4_model_data",
        )
        if r.returncode != 0:
            raise SystemExit(r.returncode)
print("Step 4 complete.")

## Pipeline Step 5: PGx analysis

Add PGx features (e.g. CPIC drug counts) to model data. Reads from Step 4 outputs and writes updated model data used by Step 6. Run for each cohort/age_band.

In [ ]:
# Pipeline Step 5: run_analysis.py for each REQUIRED_COHORTS (cohort, age_band)
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"→ Step 5: {cohort} / {age_band}")
        r = subprocess.run(
            [sys.executable, "run_analysis.py", "--cohort-name", cohort, "--age-band", age_band],
            cwd=PROJECT_ROOT / "5_pgx_analysis",
        )
        if r.returncode != 0:
            raise SystemExit(r.returncode)
print("Step 5 complete.")

## Pipeline Step 6: Final model training

Train final models per cohort/age_band. Reads Step 4 model data and Step 5 PGx features; writes trained models and `feature_schema.json` to `6_final_model/outputs` (or DATA_ROOT). These outputs are used by "Prepare models" and deployment below.

## Step 1: Generate metadata — idempotent with checkpoint

Extract valid codes (drugs, ICD, CPT) from feature importance for dashboard dropdowns. Uses Step 3b `cohort_feature_importance` when available, else Step 3 `aggregated_feature_importance`. **Checkpoint:** step is skipped if S3 checkpoint exists.

In [ ]:
# Pipeline Step 6: run_final_model.py for each REQUIRED_COHORTS (cohort, age_band)
# Note: script uses --age_band (underscore)
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"→ Step 6: {cohort} / {age_band}")
        r = subprocess.run(
            [sys.executable, "run_final_model.py", "--cohort", cohort, "--age_band", age_band],
            cwd=PROJECT_ROOT / "6_final_model",
        )
        if r.returncode != 0:
            raise SystemExit(r.returncode)
print("Step 6 complete.")

In [ ]:
import logging
logger = logging.getLogger(__name__)
if check_step_checkpoint_exists("9_dashboard_metadata", "all", "all", logger):
    print("Step 1 (generate metadata) already completed (checkpoint exists). Skipping.")
else:
    r = subprocess.run([sys.executable, "generate_metadata.py", "--all"], cwd=DATA_PREP_DIR)
    if r.returncode == 0:
        save_step_checkpoint("9_dashboard_metadata", "all", "all", logger=logger)
    if r.returncode != 0:
        raise SystemExit(r.returncode)

## Step 2: Prepare models — idempotent with checkpoint

Package models and feature schemas from `6_final_model/outputs` into `9_risk_dashboard/outputs/models`. **Checkpoint:** step is skipped if S3 checkpoint exists.

In [ ]:
if check_step_checkpoint_exists("9_dashboard_models", "all", "all", logger):
    print("Step 2 (prepare models) already completed (checkpoint exists). Skipping.")
else:
    r = subprocess.run([sys.executable, "prepare_models.py", "--all"], cwd=DATA_PREP_DIR)
    if r.returncode == 0:
        save_step_checkpoint("9_dashboard_models", "all", "all", logger=logger)
    if r.returncode != 0:
        raise SystemExit(r.returncode)

## Step 3 (optional): Combine SHAP and FFA results

For the Causal Analysis tab, combine SHAP (Step 7) and FFA (Step 8) results. Run per cohort/age_band if you have those outputs.

In [ ]:
# Optional: run for one or all cohort/age_band
# !python combine_shap_ffa_results.py --cohort opioid_ed --age-band 25-44 --output-dir "$PROJECT_ROOT/9_risk_dashboard/outputs"
# For all: implement loop or use --all-cohorts if supported
print("Optional: run combine_shap_ffa_results.py for cohorts that have SHAP/FFA outputs.")

## Step 4: Prepare Lambda directory

Assemble `lambda_dir` under `9_risk_dashboard` for Docker build (models, metadata, CPIC data).

In [ ]:
%cd "$DEPLOY_DIR"
!python prepare_lambda_dir.py

## Step 5: Verify Lambda directory

Ensure all required files are present before building the image.

In [ ]:
subprocess.run([sys.executable, "prepare_lambda_dir.py", "--verify-only"], cwd=DEPLOY_DIR, check=True)

## Step 6: Build and deploy

Build the Docker image and push to ECR; then update API Gateway/Lambda. Use the deployment script in `9_risk_dashboard/deployment`.

In [ ]:
# From 9_risk_dashboard directory:
# ./deployment/docker_build.sh
# Or manually:
# docker build -t pgx-risk-dashboard .
# Then push to ECR and update Lambda function (see docs/Step10_Results/README_results_deployment.md)
print("Run from shell: cd 9_risk_dashboard && ./deployment/docker_build.sh")
print("See: docs/Step10_Results/README_results_deployment.md")